In [ ]:
!pip3 install coinbase-advanced-py pandas pandas-ta-classic --break

In [1]:
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
from coinbase.rest import RESTClient

In [2]:
COINBASE_API_KEY = "organizations/e40f9fa4-b293-4d48-9a15-b7b99d20cfff/apiKeys/6e1e3da4-7595-46e5-9da7-4a66b3ac9267"
COINBASE_API_SECRET = "tRlwnSd3jOBlr10/zE+LQJxauK7ahkrVal4jLYzY8HORg9bzrdJGHdzF7AldfIkg9n6ak+V76aXW7R/ReT9aMQ=="

In [2]:
# Timeframes supported by Coinbase
TIMEFRAMES = {
    "1m": ("ONE_MINUTE", 60),
    "5m": ("FIVE_MINUTE", 300),
    "15m": ("FIFTEEN_MINUTE", 900),
    "30m": ("THIRTY_MINUTE", 1800),
    "1h": ("ONE_HOUR", 3600),
    "2h": ("TWO_HOUR", 7200),
    "4h": ("FOUR_HOUR", 14400),
    "6h": ("SIX_HOUR", 21600),
    "1d": ("ONE_DAY", 86400),
}
MAX_CANDLES_PER_REQUEST = 350  # Coinbase hard limit
TRADE_QUERY_LIMIT = 1000  # Fail rather than use potentially truncated trade totals.

In [4]:
# utils

def get_price(client, symbol):
    return float(client.get_product(symbol)["price"])


def get_candles(client, symbol, timeframe, count=300):
    granularity, seconds = TIMEFRAMES[timeframe]
    count = min(count, 350)
    end = int(time.time())
    start = end - count * seconds

    resp = client.get_candles(product_id=symbol, start=str(start), end=str(end), granularity=granularity)
    rows = resp.to_dict().get("candles", [])
    if not rows:
        raise RuntimeError(f"No candles returned for {symbol} {timeframe}")

    df = pd.DataFrame(rows)
    df["time"] = pd.to_datetime(df["start"].astype(int), unit="s", utc=True)
    df = df.set_index("time").drop(columns="start").astype(float)
    df = df[["open", "high", "low", "close", "volume"]].sort_index()

    # Drop the candle that is still forming
    if df.index[-1].timestamp() + seconds > time.time():
        df = df.iloc[:-1]
    return df

In [5]:
# Connect to Coinbase 
api_key = COINBASE_API_KEY
api_secret = COINBASE_API_SECRET
try:
    client = RESTClient(api_key=api_key, api_secret=api_secret)
    print(client)
except Exception as e:
    print(f"Failed to connect to Coinbase: {e}")

In [21]:
start_date = "2016-01-01"

def add_volume_delta(df):
    """Add volume_delta: buy_volume - sell_volume when available, otherwise a candle-direction estimate."""
    if {"buy_volume", "sell_volume"}.issubset(df.columns):
        df["volume_delta"] = df["buy_volume"] - df["sell_volume"]
    else:
        # volume scaled by candle body / range: negative for red candles, positive for green
        candle_range = (df["high"] - df["low"]).replace(0, np.nan)
        body_ratio = ((df["close"] - df["open"]) / candle_range).fillna(0.0)
        df["volume_delta"] = df["volume"] * body_ratio
    return df

def get_history(client, symbol, timeframe, start_date, max_empty_chunks=3):
    """Download all closed candles from start_date until now, walking backwards in API-sized chunks."""
    granularity, seconds = TIMEFRAMES[timeframe]
    floor_ts = int(pd.Timestamp(start_date, tz="UTC").timestamp())
    end, frames, empty_in_a_row, chunks = int(time.time()), [], 0, 0

    while end > floor_ts:
        start = max(end - MAX_CANDLES_PER_REQUEST * seconds, floor_ts)
        rows = []
        for attempt in range(5):
            try:
                resp = client.get_candles(product_id=symbol, start=str(start), end=str(end), granularity=granularity)
                rows = resp.to_dict().get("candles", [])
                break
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)

        if rows:
            frames.append(pd.DataFrame(rows))
            empty_in_a_row = 0
        elif frames:
            empty_in_a_row += 1
            if empty_in_a_row >= max_empty_chunks:
                break

        chunks += 1
        if chunks % 100 == 0:
            print(f"fetched back to {pd.to_datetime(start, unit='s', utc=True)}")
        end = start
        time.sleep(0.15)

    if not frames:
        raise RuntimeError(f"No candles returned for {symbol} {timeframe}")

    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["start"].astype(int), unit="s", utc=True)
    df = df.drop(columns="start").drop_duplicates(subset="time").set_index("time").sort_index().astype(float)
    df = df[["open", "high", "low", "close", "volume"]]

    # Drop the candle that is still forming
    if df.index[-1].timestamp() + seconds > time.time():
        df = df.iloc[:-1]
    return df


def add_trade_side_volumes(client, symbol, candles, seconds, limit=TRADE_QUERY_LIMIT):
    """Aggregate Coinbase trades into candle buy/sell volume using taker direction.

    Coinbase's trade `side` is the maker side, so SELL maker trades are buyer-initiated
    and BUY maker trades are seller-initiated. A full response at `limit` is treated as
    possibly truncated and raises instead of producing misleading volume totals.
    """
    result = candles.copy()
    result["buy_volume"] = 0.0
    result["sell_volume"] = 0.0

    def fetch_side_sums(start, end):
        """Return (buy, sell) for [start, end) seconds, bisecting whenever the response hits the limit."""
        response = client.get_market_trades(
            product_id=symbol, limit=limit, start=str(start), end=str(end)
        )
        trades = response.to_dict().get("trades", [])
        if len(trades) >= limit:
            if end - start <= 1:
                raise RuntimeError(
                    f"More than {limit} trades in one second for {symbol} at {start}; cannot split further."
                )
            mid = (start + end) // 2
            buy_a, sell_a = fetch_side_sums(start, mid)
            buy_b, sell_b = fetch_side_sums(mid, end)
            return buy_a + buy_b, sell_a + sell_b
        if not trades:
            return 0.0, 0.0

        trade_frame = pd.DataFrame(trades)
        trade_frame["time"] = pd.to_datetime(trade_frame["time"], utc=True)
        trade_frame["size"] = pd.to_numeric(trade_frame["size"], errors="raise")
        in_slice = (trade_frame["time"] >= pd.Timestamp(start, unit="s", tz="UTC")) & (
            trade_frame["time"] < pd.Timestamp(end, unit="s", tz="UTC")
        )
        trade_frame = trade_frame.loc[in_slice]
        maker_side = trade_frame["side"].str.upper()
        unknown_sides = set(maker_side.dropna()) - {"BUY", "SELL"}
        if maker_side.isna().any() or unknown_sides:
            raise ValueError(f"Unexpected Coinbase trade sides: {unknown_sides}")
        return (
            trade_frame.loc[maker_side == "SELL", "size"].sum(),
            trade_frame.loc[maker_side == "BUY", "size"].sum(),
        )

    for candle_time in result.index:
        start = int(candle_time.timestamp())
        buy, sell = fetch_side_sums(start, start + seconds)
        result.at[candle_time, "buy_volume"] = buy
        result.at[candle_time, "sell_volume"] = sell

    trade_totals = result["buy_volume"] + result["sell_volume"]
    matches_candle_volume = np.isclose(trade_totals, result["volume"], rtol=1e-5, atol=1e-8)
    if not matches_candle_volume.all():
        first_mismatch = result.index[~matches_candle_volume][0]
        raise RuntimeError(
            f"Trade-side volumes do not reconcile with candle volume for {symbol} at {first_mismatch}; "
            "the trade response may be incomplete, so no CSV was written."
        )

    return result

In [ ]:
"""os.makedirs("datapoints", exist_ok=True)
for symbol in ["BTC-USDT", "ETH-USDT", "SOL-USDT"]:
    os.makedirs("datapoints/" + symbol.split("-")[0], exist_ok=True)
    for timeframe in ["1d", "4h", "1h", "30m", "15m", "5m"]:
        df = get_history(client, symbol, timeframe, start_date)
        _, seconds = TIMEFRAMES[timeframe]
        df = add_trade_side_volumes(client, symbol, df, seconds)
        df = add_volume_delta(df)
        df["candle_type"] = np.where(df["close"] >= df["open"], "green", "red")
        csv_path = f"datapoints/{symbol.split('-')[0]}/{symbol}_{timeframe}.csv"
        df.to_csv(csv_path)
        print(f"Saved {csv_path} with columns: {', '.join(df.columns)}")"""

In [ ]:
import dukascopy_python as dk
from dukascopy_python.instruments import INSTRUMENT_FX_METALS_XAU_USD, INSTRUMENT_ETF_CFD_US_QQQ_US_USD, INSTRUMENT_FX_CROSSES_EUR_GBP, INSTRUMENT_FX_CROSSES_EUR_JPY, INSTRUMENT_FX_MAJORS_EUR_USD, INSTRUMENT_FX_MAJORS_GBP_USD, INSTRUMENT_FX_MAJORS_USD_JPY, INSTRUMENT_IDX_AMERICA_E_D_J_IND, INSTRUMENT_IDX_AMERICA_E_NQ_100, INSTRUMENT_IDX_AMERICA_E_SANDP_500, INSTRUMENT_US_AMD_US_USD, INSTRUMENT_US_AMZN_US_USD, INSTRUMENT_US_AAPL_US_USD, INSTRUMENT_US_GOOGL_US_USD, INSTRUMENT_US_HPE_US_USD, INSTRUMENT_US_MSFT_US_USD, INSTRUMENT_VCCY_BTC_USD, INSTRUMENT_VCCY_ETH_USD
from pathlib import Path

def add_volume_delta(df):
    """Dukascopy has no buy/sell split, so estimate: volume * (close - open) / (high - low)."""
    candle_range = (df["high"] - df["low"]).replace(0, np.nan)
    body_ratio = ((df["close"] - df["open"]) / candle_range).fillna(0.0)
    df["volume_delta"] = df["volume"] * body_ratio
    return df

start, end = pd.Timestamp("2016-01-01"), pd.Timestamp.now()

# Download year by year to keep each request small
os.makedirs("datapoints", exist_ok=True)
for symbol in [INSTRUMENT_FX_METALS_XAU_USD, INSTRUMENT_ETF_CFD_US_QQQ_US_USD, INSTRUMENT_FX_CROSSES_EUR_GBP, INSTRUMENT_FX_CROSSES_EUR_JPY, INSTRUMENT_FX_MAJORS_EUR_USD, INSTRUMENT_FX_MAJORS_GBP_USD, INSTRUMENT_FX_MAJORS_USD_JPY, INSTRUMENT_IDX_AMERICA_E_D_J_IND, INSTRUMENT_IDX_AMERICA_E_NQ_100, INSTRUMENT_IDX_AMERICA_E_SANDP_500, INSTRUMENT_US_AMD_US_USD, INSTRUMENT_US_AMZN_US_USD, INSTRUMENT_US_AAPL_US_USD, INSTRUMENT_US_GOOGL_US_USD, INSTRUMENT_US_HPE_US_USD, INSTRUMENT_US_MSFT_US_USD, INSTRUMENT_VCCY_BTC_USD, INSTRUMENT_VCCY_ETH_USD]:
    Path("datapoints/" + symbol.split("/")[0].upper()).mkdir(exist_ok=True)
    for timeframe in [dk.INTERVAL_MIN_1, dk.INTERVAL_MIN_5, dk.INTERVAL_MIN_15, dk.INTERVAL_MIN_30, dk.INTERVAL_HOUR_1, dk.INTERVAL_HOUR_4][::-1]:
        parts = []
        for year in range(start.year, end.year + 1):
            s = max(start, pd.Timestamp(year, 1, 1))
            e = min(end, pd.Timestamp(year + 1, 1, 1))
            parts.append(dk.fetch(symbol, timeframe, dk.OFFER_SIDE_BID, s.to_pydatetime(), e.to_pydatetime()))

        df = pd.concat(parts)
        df = df[~df.index.duplicated()].sort_index()
        df.index.name = "time"
        df = add_volume_delta(df)
        df["candle_type"] = np.where(df["close"] >= df["open"], "green", "red")
        time_name = ""
        if timeframe == dk.INTERVAL_MIN_1:
            time_name = "1m"
        elif timeframe == dk.INTERVAL_MIN_5:
            time_name = "5m"
        elif timeframe == dk.INTERVAL_MIN_15:
            time_name = "15m"
        elif timeframe == dk.INTERVAL_MIN_30:
            time_name = "30m"
        elif timeframe == dk.INTERVAL_HOUR_1:
            time_name = "1h"
        elif timeframe == dk.INTERVAL_HOUR_4:
            time_name = "4h"
        df.to_csv(f"datapoints/{symbol.split('/')[0].upper()}/{symbol.split('/')[0].upper()}_{time_name}.csv")